# BTC limit-order-book direction prediction — full pipeline

One notebook for the whole project. Every step reads its inputs from and writes its outputs to
one DuckDB database (`lob.duckdb`), so steps can be run separately, resumed after a disconnect,
and inspected with plain SQL.

| Section | What it does | Writes to the database |
|---|---|---|
| 0 Setup | paths, switches, constants, hyperparameters | – |
| 1 Database | small helpers around DuckDB | – |
| 2 Features | raw CSV → features, labels, z-score, purged split, PCA | `features`, `meta` |
| 3 Backtest | trading rules and the fee-aware, non-overlapping backtest | – |
| 4 Models | CNN, BiLSTM, Transformer (PyTorch) and LightGBM | – |
| 5 Experiment runner | train → choose rules on validation → test once; optional import of earlier results | `runs`, `predictions`, `history` |
| 6 Run experiments | main split, horizons, walk-forward | same |
| 7 Analysis | robustness, execution delay, BTC/ETH timestamps | `robustness`, `delay`, `timestamp_check` |
| 8 Report | tables, figures and CSV exports | files in `EXPORT_DIR` |
#
**Database tables**
- `features` – one row per second: `row_id`, `time`, main `split`, labels / returns / target flags per horizon, all feature columns
- `meta` – key/value JSON (PCA columns, explained variance, label distribution, …)
- `runs` – one row per trained model: identifiers (`scheme`, `fold`, `model`, `feature_set`, `horizon`, `seed`) and all test metrics
- `predictions` – class probabilities for every validation and test target of every run
- `history` – training curves per run and epoch
- `robustness`, `delay`, `timestamp_check` – results of the checks in Section 7

## 0 · Setup
Set the switches in the next cell, then *Run all*. Steps whose results are already in the
database are skipped, so re-running is cheap.

In [ ]:
import os, sys, json, gc, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q kagglehub duckdb lightgbm pyarrow

# ── Where things live ────────────────────────────────────────────────────────
WORK_DIR = "/content/drive/MyDrive/mat629-lob" if IN_COLAB else os.path.abspath("work")
SUBSET_ROWS = None      # e.g. 100_000 (≈1.2 days) for a quick development run; None = all data
RAW_DIR = None          # folder with BTC_1sec.csv / ETH_1sec.csv; None = download with kagglehub
DB_PATH = os.path.join(WORK_DIR, "lob.duckdb" if SUBSET_ROWS is None else f"lob_subset{SUBSET_ROWS}.duckdb")
EXPORT_DIR = os.path.join(WORK_DIR, "export")   # report figures + CSV snapshots for GitHub
os.makedirs(WORK_DIR, exist_ok=True)

# ── What to run ──────────────────────────────────────────────────────────────
STEPS = {
    "import_legacy": False,  # Section A: import results computed before this notebook (run once)
    "build_features": True,  # Section 2 (skipped if the table exists)
    "main_split": True,      # Section 6.1: all models × feature sets, 10 s
    "horizons": True,        # Section 6.2: Full feature set at 30/60/300 s (+ LightGBM all sets)
    "walk_forward": True,    # Section 6.3: CNN (3 seeds + ensemble) and LightGBM, 6 folds
    "analysis": True,        # Section 7
    "report": True,          # Section 8
}
MODELS = ["cnn", "lstm", "transformer", "lgbm"]   # models to train in Section 6
# On macOS, PyTorch and LightGBM crash when loaded in the same process (on Colab/Linux they
# don't). Locally, run the neural models and LightGBM in separate kernel sessions by editing MODELS.
print("Database:", DB_PATH)

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
SEED = 42
KAGGLE_DATASET = "martinsn/high-frequency-crypto-limit-order-book-data"

HORIZON = 10                                   # main horizon (seconds)
HORIZONS = (10, 30, 60, 300)                   # labels are built for all of these
LABEL_THRESHOLDS = {10: 1.5e-4, 30: 3e-4, 60: 3e-4, 300: 3e-4}   # 10 s: 1.5 bp; longer: round-trip cost

N_LEVELS = 15
WINDOW_SHORT, WINDOW_LONG = 10, 60             # seconds
ZSCORE_WINDOW, ZSCORE_MIN_PERIODS = 3600, 60
MAX_PCA_COMPONENTS = 10
TICK_SIZE = 0.01
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15              # test = remaining 0.15

FEE = 1.5e-4                                   # per unit of position change
SECONDS_PER_YEAR = 365 * 24 * 3600
MIN_TURNOVER = 20                              # rules that barely trade on validation are not eligible

FEATURE_SETS = ["full", "no_pca", "no_market"]
FEATURE_SET_DESC = {"full": "PCA(LOB) + market + classic", "no_pca": "raw LOB + market + classic",
                    "no_market": "PCA(LOB) + classic"}
CLASS_NAMES = ["Down", "Flat", "Up"]           # model classes 0 / 1 / 2 ↔ labels -1 / 0 / +1

# ── Hyperparameters ──────────────────────────────────────────────────────────
HPARAMS = {
    "cnn": dict(seq_len=10, batch_size=256,
                model_kwargs=dict(n_filters=64, kernel_size=3, n_blocks=2, dropout=0.3),
                train_kwargs=dict(epochs=50, lr=1e-4, weight_decay=1e-2, patience=10, scheduler="cosine")),
    "lstm": dict(seq_len=10, batch_size=256,
                 model_kwargs=dict(hidden_size=32, n_layers=2, dropout=0.4),
                 train_kwargs=dict(epochs=40, lr=1e-4, weight_decay=2e-3, patience=6, scheduler="cosine")),
    "transformer": dict(seq_len=30, batch_size=512,
                        model_kwargs=dict(d_model=64, n_heads=4, n_layers=3, d_ff=128, dropout=0.1),
                        train_kwargs=dict(epochs=30, lr=1e-3, weight_decay=1e-4, patience=6, scheduler="onecycle")),
    "lgbm": dict(params=dict(n_estimators=2000, learning_rate=0.05, num_leaves=31, min_child_samples=200,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1),
                 early_stopping_rounds=50),
}
EPOCHS_OVERRIDE = None    # e.g. 1 for a smoke test
NUM_WORKERS = 2 if IN_COLAB else 0

## 1 · Database
Short-lived connections, so a crash or disconnect never leaves the file locked.

In [ ]:
import duckdb

def connect():
    con = duckdb.connect(DB_PATH)
    con.execute("SET TimeZone = 'UTC'")      # timestamps come back in UTC, not local time
    return con

def db_query(sql, params=None):
    """Run a query and return a DataFrame."""
    with connect() as con:
        return con.execute(sql, params or []).df()

def db_exec(sql, params=None):
    with connect() as con:
        con.execute(sql, params or [])

def db_has_table(name):
    with connect() as con:
        return con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name = ?",
                           [name]).fetchone()[0] > 0

def db_write(name, frame, replace=False):
    """Append `frame` to table `name` (matching columns by name); create it if missing."""
    with connect() as con:
        con.register("frame", frame)
        exists = con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name = ?",
                             [name]).fetchone()[0] > 0
        if replace or not exists:
            con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM frame")
        else:
            con.execute(f"INSERT INTO {name} BY NAME SELECT * FROM frame")
        con.unregister("frame")

def meta_get(key, default=None):
    if not db_has_table("meta"):
        return default
    r = db_query("SELECT value FROM meta WHERE key = ?", [key])
    return json.loads(r["value"].iloc[0]) if len(r) else default

def meta_set(**items):
    frame = pd.DataFrame({"key": list(items), "value": [json.dumps(v, default=str) for v in items.values()]})
    if db_has_table("meta"):
        db_exec("DELETE FROM meta WHERE key IN (SELECT unnest(?))", [list(items)])
    db_write("meta", frame)

def done_run_ids():
    return set(db_query("SELECT run_id FROM runs")["run_id"]) if db_has_table("runs") else set()

RUN_ID_COLS = {"run_id": "VARCHAR PRIMARY KEY", "scheme": "VARCHAR", "fold": "INTEGER", "test_day": "VARCHAR",
               "model": "VARCHAR", "feature_set": "VARCHAR", "horizon": "INTEGER", "seed": "VARCHAR",
               "best_epoch": "INTEGER"}
METRIC_COLS = ["test_acc", "flat_acc", "test_dir_f1", "argmax_return", "argmax_sharpe", "argmax_turnover",
               "threshold", "th_return", "th_sharpe", "th_mdd", "th_turnover", "hy_enter", "hy_exit", "hy_return",
               "hy_sharpe", "hy_mdd", "hy_turnover", "bh_return"]
# `runs` gets an explicit schema so that a first row with missing metrics cannot fix a wrong column type
db_exec("CREATE TABLE IF NOT EXISTS runs (" + ", ".join(
    [f"{c} {t}" for c, t in RUN_ID_COLS.items()] + [f"{c} DOUBLE" for c in METRIC_COLS]
    + ["n_test_steps INTEGER", "created_at TIMESTAMPTZ"]) + ")")
db_exec("""CREATE TABLE IF NOT EXISTS history (run_id VARCHAR, epoch INTEGER, train_loss DOUBLE, val_loss DOUBLE,
                                             val_acc DOUBLE, val_dir_f1 DOUBLE)""")

def db_overview():
    if not os.path.exists(DB_PATH):
        return pd.DataFrame()
    tables = db_query("SELECT table_name FROM information_schema.tables ORDER BY table_name")["table_name"]
    return pd.DataFrame({"table": tables,
                         "rows": [db_query(f"SELECT count(*) AS n FROM {t}")["n"].iloc[0] for t in tables]})

db_overview()

## 2 · Features
load + align BTC/ETH on the second → market features → labels (by timestamp) → classic LOB
factors → backward-looking rolling z-score → chronological 70/15/15 split with purging → PCA
fitted on training rows only. Result: table `features` (+ `meta`).

In [ ]:
EPS = 1e-8
LOB_COLS = [f"{side}_{kind}_{i}" for i in range(N_LEVELS)
            for kind, side in [("distance", "bids"), ("distance", "asks"), ("notional", "bids"), ("notional", "asks")]]
MARKET_COLS = [
    "spread", "btc_oimb_1", "btc_volatility_10s", "btc_volatility_60s", "btc_momentum_10s", "btc_momentum_60s",
    "eth_spread", "eth_oimb_1", "eth_volatility_10s", "eth_volatility_60s", "eth_momentum_10s", "eth_momentum_60s",
    "btc_eth_momentum_diff_10s", "btc_eth_momentum_diff_60s", "btc_eth_vol_ratio_10s",
]
CLASSIC_COLS = [
    "factor_queue_imbalance", "factor_ofi", "factor_ofi_10s", "factor_ofi_60s", "factor_kyle_lambda",
    "factor_depth_ratio", "factor_depth_imb", "factor_depth_ratio_smooth",
    "factor_tick_sign", "factor_tick_momentum_10s", "factor_tick_momentum_60s",
    "factor_amihud", "factor_amihud_60s",
]

def find_raw_dir(raw_dir=None):
    if raw_dir is not None:
        return raw_dir
    import kagglehub
    return kagglehub.dataset_download(KAGGLE_DATASET)

def read_snapshots(path, cols, nrows):
    # prices / best-level distances stay float64 (to rebuild exact tick prices for OFI)
    keep64 = {"midpoint", "bids_distance_0", "asks_distance_0"}
    dtype = {c: ("float64" if c in keep64 else "float32") for c in cols if c != "system_time"}
    d = pd.read_csv(path, usecols=cols, dtype=dtype, nrows=nrows)
    d["time"] = pd.to_datetime(d.pop("system_time"), utc=True).dt.floor("s")
    return d.drop_duplicates("time", keep="last")

def load_raw(raw_dir=None, nrows=None):
    raw_dir = find_raw_dir(raw_dir)
    btc = read_snapshots(os.path.join(raw_dir, "BTC_1sec.csv"), ["system_time", "midpoint", "spread", *LOB_COLS], nrows)
    eth = read_snapshots(os.path.join(raw_dir, "ETH_1sec.csv"),
                         ["system_time", "midpoint", "spread", "bids_notional_0", "asks_notional_0"], nrows)
    eth = eth.rename(columns=lambda c: c if c == "time" else f"eth_{c}")
    d = btc.merge(eth, on="time", how="inner").sort_values("time").reset_index(drop=True)
    print(f"BTC {len(btc):,} | ETH {len(eth):,} | merged {len(d):,} rows | {d['time'].iloc[0]} → {d['time'].iloc[-1]}")
    return d

def add_market_features(d):
    ws, wl = WINDOW_SHORT, WINDOW_LONG
    for asset, mid, b0, a0 in [("btc", "midpoint", "bids_notional_0", "asks_notional_0"),
                               ("eth", "eth_midpoint", "eth_bids_notional_0", "eth_asks_notional_0")]:
        r = np.log(d[mid] / d[mid].shift(1))
        d[f"{asset}_log_return"] = r
        d[f"{asset}_oimb_1"] = (d[b0] - d[a0]) / (d[b0] + d[a0] + EPS)
        d[f"{asset}_volatility_10s"] = r.rolling(ws).std()
        d[f"{asset}_volatility_60s"] = r.rolling(wl).std()
        d[f"{asset}_momentum_10s"] = r.rolling(ws).sum()
        d[f"{asset}_momentum_60s"] = r.rolling(wl).sum()
    d["btc_eth_momentum_diff_10s"] = d["btc_momentum_10s"] - d["eth_momentum_10s"]
    d["btc_eth_momentum_diff_60s"] = d["btc_momentum_60s"] - d["eth_momentum_60s"]
    d["btc_eth_vol_ratio_10s"] = d["btc_volatility_10s"] / (d["eth_volatility_10s"] + EPS)
    return d

def add_labels(d):
    """future_return_h[t] = mid(t + h s) / mid(t) − 1, looked up by timestamp (the feed has gaps)."""
    mid = d.set_index("time")["midpoint"]
    cols = {}
    for h in HORIZONS:
        ret = mid.reindex(d["time"] + pd.Timedelta(seconds=h)).to_numpy() / d["midpoint"].to_numpy() - 1
        th = LABEL_THRESHOLDS[h]
        cols[f"future_return_{h}"] = ret.astype("float32")
        cols[f"label_{h}"] = np.select([ret > th, ret < -th], [1, -1], default=0).astype("int8")
    return pd.concat([d, pd.DataFrame(cols, index=d.index)], axis=1)

def add_classic_factors(d):
    ws, wl = WINDOW_SHORT, WINDOW_LONG
    r = d["btc_log_return"]
    # 1. weighted 5-level queue imbalance (Cont, Kukanov & Stoikov 2014)
    w = np.exp(-0.5 * np.arange(5)); w /= w.sum()
    bid = d[[f"bids_notional_{i}" for i in range(5)]].to_numpy()
    ask = d[[f"asks_notional_{i}" for i in range(5)]].to_numpy()
    d["factor_queue_imbalance"] = ((bid - ask) / (bid + ask + EPS) * w).sum(axis=1)
    # 2. order flow imbalance; distances are relative to the mid, so rebuild tick prices first
    bp = (d["midpoint"] * (1 + d["bids_distance_0"]) / TICK_SIZE).round()
    ap = (d["midpoint"] * (1 + d["asks_distance_0"]) / TICK_SIZE).round()
    bv, av = d["bids_notional_0"], d["asks_notional_0"]
    e_bid = bv * (bp >= bp.shift(1)) - bv.shift(1) * (bp <= bp.shift(1))
    e_ask = av.shift(1) * (ap >= ap.shift(1)) - av * (ap <= ap.shift(1))
    d["factor_ofi"] = e_bid - e_ask
    d["factor_ofi_10s"] = d["factor_ofi"].rolling(ws).sum()
    d["factor_ofi_60s"] = d["factor_ofi"].rolling(wl).sum()
    # 3. Kyle's lambda: rolling slope of return on OFI
    ofi = d["factor_ofi"].fillna(0)
    d["factor_kyle_lambda"] = r.rolling(wl).cov(ofi) / (ofi.rolling(wl).var() + EPS)
    # 4. depth ratio / imbalance over 10 levels
    tb = d[[f"bids_notional_{i}" for i in range(10)]].sum(axis=1)
    ta = d[[f"asks_notional_{i}" for i in range(10)]].sum(axis=1)
    d["factor_depth_ratio"] = ta / (tb + EPS)
    d["factor_depth_imb"] = (tb - ta) / (tb + ta + EPS)
    d["factor_depth_ratio_smooth"] = d["factor_depth_ratio"].rolling(ws).mean()
    # 5. tick rule on mid-price changes
    tick = np.sign(r.fillna(0)).replace(0, np.nan).ffill().fillna(1)
    d["factor_tick_sign"] = tick
    d["factor_tick_momentum_10s"] = tick.rolling(ws).sum()
    d["factor_tick_momentum_60s"] = tick.rolling(wl).sum()
    # 6. Amihud-style illiquidity, top-5 book depth standing in for traded volume
    depth5 = (d[[f"bids_notional_{i}" for i in range(5)]].sum(axis=1)
              + d[[f"asks_notional_{i}" for i in range(5)]].sum(axis=1))
    d["factor_amihud"] = r.abs() / (depth5 + EPS)
    d["factor_amihud_60s"] = d["factor_amihud"].rolling(wl).mean()
    return d

def rolling_zscore(d, cols, clip=10.0, chunk=16):
    """Backward-looking z-score over the last ZSCORE_WINDOW rows (incl. the current one).
    No epsilon in the denominator (Amihud / Kyle λ are ~1e-10); zero std → 0; clipped to ±clip."""
    for i in range(0, len(cols), chunk):
        c = cols[i:i + chunk]
        x = d[c].astype("float64")
        roll = x.rolling(ZSCORE_WINDOW, min_periods=ZSCORE_MIN_PERIODS)
        mean, std = roll.mean(), roll.std()
        z = ((x - mean) / std.where(std > 0)).fillna(0.0).where(mean.notna())
        d[c] = z.clip(-clip, clip).astype("float32")
        del x, roll, mean, std, z; gc.collect()
    return d

def assign_splits(d):
    """Chronological 70/15/15 split. A row is a target for horizon h only if [t, t+h] stays in its split."""
    n = len(d)
    cut1, cut2 = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
    split = np.empty(n, dtype=object)
    split[:cut1], split[cut1:cut2], split[cut2:] = "train", "val", "test"
    d["split"] = split
    next_start = d["split"].map({"train": d["time"].iloc[cut1], "val": d["time"].iloc[cut2]})
    for h in HORIZONS:
        label_end = d["time"] + pd.Timedelta(seconds=h)
        d[f"is_target_{h}"] = d[f"future_return_{h}"].notna() & ((label_end < next_start) | next_start.isna())
    for s in ["train", "val", "test"]:
        m = d["split"] == s
        print(f"  {s:<5} {m.sum():>9,} rows | {d.loc[m, 'time'].iloc[0]} → {d.loc[m, 'time'].iloc[-1]}")
    return d

def fit_pca(train_lob, n_keep=None):
    """StandardScaler + PCA on training LOB rows; keep Kaiser components (λ > 1), capped."""
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler().fit(train_lob)
    pca = PCA(n_components=0.90, random_state=SEED).fit(scaler.transform(train_lob))
    if n_keep is None:
        n_keep = int(min(max((pca.explained_variance_ > 1).sum(), 1), MAX_PCA_COMPONENTS))
    return scaler, pca, n_keep

def build_features():
    d = load_raw(RAW_DIR, SUBSET_ROWS)
    d = add_classic_factors(add_labels(add_market_features(d)))
    feat = LOB_COLS + MARKET_COLS + CLASSIC_COLS
    d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=feat).reset_index(drop=True)
    d = rolling_zscore(d, feat)
    d = d.dropna(subset=feat).reset_index(drop=True)
    d = assign_splits(d)

    train = (d["split"] == "train").to_numpy()
    scaler, pca, k = fit_pca(d.loc[train, LOB_COLS])
    pca_cols = [f"pca_{i + 1}" for i in range(k)]
    Z = pca.transform(scaler.transform(d[LOB_COLS]))[:, :k].astype("float32")
    d = pd.concat([d, pd.DataFrame(Z, columns=pca_cols, index=d.index)], axis=1)
    print(f"  PCA: {pca.n_components_} comps for 90% → keep {k} ({pca.explained_variance_ratio_[:k].sum():.3f} variance)")

    label_cols = [f"{p}_{h}" for h in HORIZONS for p in ("is_target", "label", "future_return")]
    out = d[["time", "split"] + label_cols + pca_cols + feat].copy()
    out.insert(0, "row_id", np.arange(len(out), dtype=np.int64))
    db_write("features", out, replace=True)
    meta_set(pca_cols=pca_cols, n_rows=len(out), horizons=list(HORIZONS), label_thresholds=LABEL_THRESHOLDS,
             pca_explained_variance=pca.explained_variance_ratio_.round(5).tolist(), pca_kept=k,
             label_dist={h: out.loc[out[f"is_target_{h}"], f"label_{h}"].value_counts(normalize=True)
                            .sort_index().round(4).to_dict() for h in HORIZONS})
    print(f"Saved features: {out.shape}")

def import_legacy_features(root=WORK_DIR):
    """Use data/features.parquet from the earlier script-based version instead of rebuilding."""
    pq = os.path.join(root, "data", "features.parquet")
    if not os.path.exists(pq):
        return False
    f = pd.read_parquet(pq)
    f.insert(0, "row_id", np.arange(len(f), dtype=np.int64))
    db_write("features", f, replace=True)
    with open(os.path.splitext(pq)[0] + "_meta.json") as fh:
        m = json.load(fh)
    meta_set(pca_cols=m["pca_cols"], n_rows=len(f), horizons=m.get("horizons", list(HORIZONS)),
             label_dist=m.get("label_dist"), pca=m.get("pca"))
    print(f"Imported features {f.shape} from {pq}")
    return True

In [ ]:
if not db_has_table("features"):
    if not (STEPS["import_legacy"] and import_legacy_features()) and STEPS["build_features"]:
        build_features()
db_overview()

### Load the feature table
Everything below works on `df` (one row per second) and `META`.

In [ ]:
df = db_query("SELECT * FROM features ORDER BY row_id")
META = {k: json.loads(v) for k, v in db_query("SELECT key, value FROM meta").itertuples(index=False)}
PCA_COLS = META["pca_cols"]
for h in HORIZONS:
    df[f"is_target_{h}"] = df[f"is_target_{h}"].astype(bool)
print(f"{len(df):,} rows | {df['time'].iloc[0]} → {df['time'].iloc[-1]} | PCA columns: {len(PCA_COLS)}")

def feature_columns(feature_set):
    groups = {"full": [PCA_COLS, MARKET_COLS, CLASSIC_COLS], "no_pca": [LOB_COLS, MARKET_COLS, CLASSIC_COLS],
              "no_market": [PCA_COLS, CLASSIC_COLS]}
    return [c for g in groups[feature_set] for c in g]

def target_indices(frame, split, seq_len=1, horizon=HORIZON):
    idx = np.flatnonzero(((frame["split"] == split) & frame[f"is_target_{horizon}"]).to_numpy())
    return idx[idx >= seq_len - 1]

def class_labels(frame, horizon=HORIZON):
    return (frame[f"label_{horizon}"].to_numpy() + 1).astype(np.int64)

## 3 · Backtest
* A position in {−1, 0, +1} (full capital) is opened at a target row and held for exactly *h* seconds.
* Steps are non-overlapping: the next one starts ≥ *h* rows (≥ *h* seconds) later.
* Cost = 1.5 bp × |position change| (a reversal pays twice; the first entry is charged).
* Trading rules: **argmax**; **threshold** θ (long if P(Up) > θ, short if P(Down) > θ);
  **hysteresis** (enter above `enter`, hold while ≥ `exit`, reverse on a strong opposite signal).
  θ and (enter, exit) are chosen on **validation** only.
* Sharpe is annualised with √(steps per year) ≈ 1776 at 10 s — compare across models only.

In [ ]:
def annual_factor(horizon=HORIZON):
    return SECONDS_PER_YEAR / horizon

def nonoverlap_steps(rows, horizon=HORIZON):
    """Positions into sorted `rows` such that each kept row is ≥ horizon rows after the previous one."""
    keep, nxt = [], -np.inf
    for i, r in enumerate(rows):
        if r >= nxt:
            keep.append(i); nxt = r + horizon
    return np.asarray(keep, dtype=np.int64)

def positions_from_probs(probs, threshold=None):
    if threshold is None:
        return probs.argmax(1) - 1
    pos = np.zeros(len(probs), dtype=np.int64)
    pos[probs[:, 0] > threshold] = -1
    pos[probs[:, 2] > threshold] = 1
    return pos

def positions_hysteresis(probs, enter, exit_):
    p_down, p_up = probs[:, 0], probs[:, 2]
    pos = np.zeros(len(probs), dtype=np.int64)
    cur = 0
    for i in range(len(probs)):
        if cur >= 0 and p_down[i] > enter and p_down[i] > p_up[i]:
            cur = -1
        elif cur <= 0 and p_up[i] > enter and p_up[i] > p_down[i]:
            cur = 1
        elif cur == 1 and p_up[i] < exit_:
            cur = 0
        elif cur == -1 and p_down[i] < exit_:
            cur = 0
        pos[i] = cur
    return pos

def backtest(pos, returns, fee=FEE, horizon=HORIZON):
    pos = np.asarray(pos, dtype=np.float64); returns = np.asarray(returns, dtype=np.float64)
    trades = np.abs(np.diff(pos, prepend=0.0))
    net = pos * returns - fee * trades
    equity = np.cumprod(1 + net)
    std = net.std(); active = pos != 0
    return {"total_return": float(equity[-1] - 1),
            "sharpe": float(net.mean() / std * np.sqrt(annual_factor(horizon))) if std > 0 else 0.0,
            "max_drawdown": float((equity / np.maximum.accumulate(equity) - 1).min()),
            "win_rate": float((net[active] > 0).mean()) if active.any() else float("nan"),
            "exposure": float(active.mean()), "turnover": float(trades.sum()),
            "n_steps": int(len(pos)), "equity": equity, "net": net}

def fmt_stats(name, s):
    return (f"{name:<28} return {s['total_return']:>8.2%} | Sharpe {s['sharpe']:>8.2f} | MDD {s['max_drawdown']:>8.2%}"
            f" | exposure {s['exposure']:>6.1%} | turnover {s['turnover']:>7,.0f}")

def sweep_thresholds(probs, returns, horizon):
    rows = []
    for th in np.round(np.linspace(0.35, 0.90, 23), 4):
        s = backtest(positions_from_probs(probs, th), returns, FEE, horizon)
        rows.append({"threshold": th, "sharpe": s["sharpe"], "total_return": s["total_return"], "turnover": s["turnover"]})
    return pd.DataFrame(rows)

def sweep_hysteresis(probs, returns, horizon):
    rows = []
    for enter in np.round(np.arange(0.40, 0.901, 0.05), 2):
        for exit_ in np.round(np.arange(0.0, enter - 0.049, 0.05), 2):
            s = backtest(positions_hysteresis(probs, enter, exit_), returns, FEE, horizon)
            rows.append({"enter": enter, "exit": exit_, "sharpe": s["sharpe"],
                         "total_return": s["total_return"], "turnover": s["turnover"]})
    return pd.DataFrame(rows)

def pick_best(sweep):
    eligible = sweep[sweep["turnover"] >= MIN_TURNOVER]
    return (eligible if len(eligible) else sweep).sort_values("sharpe").iloc[-1]

def test_steps(test_rows, horizon):
    """Row ids of the non-overlapping test steps (and their positions in test_rows)."""
    st = nonoverlap_steps(test_rows, horizon)
    return st, test_rows[st]

## 4 · Models
Three PyTorch sequence classifiers (input: the last *T* seconds of features → logits for
Down / Flat / Up) and a LightGBM baseline that sees only the current second.

In [ ]:
NEED_TORCH = any(m != "lgbm" for m in MODELS) and any(STEPS[k] for k in ("main_split", "horizons", "walk_forward"))
if NEED_TORCH:
    import math
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("PyTorch device:", DEVICE)

    class ResidualBlock1D(nn.Module):
        def __init__(self, in_ch, out_ch, kernel_size=3, dropout=0.3):
            super().__init__()
            pad = kernel_size // 2
            self.conv1, self.bn1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad), nn.BatchNorm1d(out_ch)
            self.conv2, self.bn2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad), nn.BatchNorm1d(out_ch)
            self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
            self.act, self.dropout = nn.GELU(), nn.Dropout(dropout)

        def forward(self, x):
            out = self.act(self.bn1(self.conv1(x)))
            return self.dropout(self.act(self.bn2(self.conv2(out)) + self.skip(x)))

    class LOBCNN(nn.Module):
        """1×1 projection → residual Conv1d blocks → last time step → MLP head."""
        def __init__(self, n_features, n_filters=64, kernel_size=3, n_blocks=2, dropout=0.3):
            super().__init__()
            self.input_proj = nn.Sequential(nn.Conv1d(n_features, n_filters, 1), nn.BatchNorm1d(n_filters), nn.GELU())
            blocks, ch = [], n_filters
            for i in range(n_blocks):
                out_ch = min(n_filters * 2 ** i, n_filters * 4)
                blocks.append(ResidualBlock1D(ch, out_ch, kernel_size, dropout)); ch = out_ch
            self.res_blocks = nn.Sequential(*blocks)
            self.head = nn.Sequential(nn.Linear(ch, ch // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(ch // 2, 3))
            for m in self.modules():
                if isinstance(m, nn.Conv1d):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)
                elif isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

        def forward(self, x):
            return self.head(self.res_blocks(self.input_proj(x.permute(0, 2, 1)))[:, :, -1])

    class LOBLSTM(nn.Module):
        """2-layer bidirectional LSTM → LayerNorm on the last forward/backward states → MLP head."""
        def __init__(self, n_features, hidden_size=32, n_layers=2, dropout=0.4):
            super().__init__()
            self.lstm = nn.LSTM(n_features, hidden_size, num_layers=n_layers, batch_first=True,
                                bidirectional=True, dropout=dropout if n_layers > 1 else 0.0)
            self.norm = nn.LayerNorm(hidden_size * 2)
            self.head = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.GELU(), nn.Dropout(dropout),
                                      nn.Linear(hidden_size, 3))
            for name, p in self.lstm.named_parameters():
                if "weight_ih" in name:
                    nn.init.xavier_uniform_(p)
                elif "weight_hh" in name:
                    nn.init.orthogonal_(p)
                elif "bias" in name:
                    nn.init.zeros_(p)
                    with torch.no_grad():
                        p[p.size(0) // 4: p.size(0) // 2].fill_(1.0)   # forget-gate bias
            for m in self.head.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

        def forward(self, x):
            _, (h_n, _) = self.lstm(x)
            return self.head(self.norm(torch.cat((h_n[-2], h_n[-1]), dim=1)))

    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=512, dropout=0.1):
            super().__init__()
            self.dropout = nn.Dropout(dropout)
            pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe = torch.zeros(max_len, d_model)
            pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
            self.register_buffer("pe", pe.unsqueeze(0))

        def forward(self, x):
            return self.dropout(x + self.pe[:, : x.size(1)])

    class LOBTransformer(nn.Module):
        """Linear projection + sinusoidal positions → pre-norm encoder → last token → MLP head."""
        def __init__(self, n_features, d_model=64, n_heads=4, n_layers=3, d_ff=128, dropout=0.1):
            super().__init__()
            self.proj = nn.Linear(n_features, d_model)
            self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
            layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, activation="gelu",
                                               batch_first=True, norm_first=True)
            self.encoder = nn.TransformerEncoder(layer, n_layers, norm=nn.LayerNorm(d_model), enable_nested_tensor=False)
            self.head = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout),
                                      nn.Linear(d_model // 2, 3))
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

        def forward(self, x):
            return self.head(self.encoder(self.pos_enc(self.proj(x)))[:, -1, :])

    TORCH_MODELS = {"cnn": LOBCNN, "lstm": LOBLSTM, "transformer": LOBTransformer}

    class WindowDataset(Dataset):
        """Sample i = (features of rows t−T+1 … t, label of row t), t = targets[i]."""
        def __init__(self, X, y, targets, seq_len):
            self.X, self.y, self.seq_len = X, y, seq_len
            self.targets = torch.as_tensor(targets, dtype=torch.long)
        def __len__(self):
            return len(self.targets)
        def __getitem__(self, i):
            t = int(self.targets[i])
            return self.X[t - self.seq_len + 1: t + 1], self.y[t]

    def set_seed(seed):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    @torch.no_grad()
    def predict_proba(model, loader):
        model.eval()
        probs, labels = [], []
        for Xb, yb in loader:
            probs.append(torch.softmax(model(Xb.to(DEVICE, non_blocking=True)), dim=-1).cpu()); labels.append(yb)
        return torch.cat(probs).numpy(), torch.cat(labels).numpy()

    def fit_torch(model, loaders, class_w, epochs, lr, weight_decay, patience, scheduler):
        """Train; early-stop on validation loss and restore the best weights. Returns the history."""
        from sklearn.metrics import f1_score
        crit = nn.CrossEntropyLoss(weight=class_w)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        if scheduler == "cosine":
            sched, per_batch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs), False
        else:
            sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, pct_start=0.1,
                                                        steps_per_epoch=len(loaders["train"]), anneal_strategy="cos")
            per_batch = True
        hist, best, best_state, bad = [], float("inf"), None, 0
        for epoch in range(1, epochs + 1):
            model.train(); run, n = 0.0, 0
            for Xb, yb in loaders["train"]:
                Xb, yb = Xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                opt.zero_grad(); loss = crit(model(Xb), yb); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                if per_batch: sched.step()
                run += loss.item() * len(yb); n += len(yb)
            if not per_batch: sched.step()
            model.eval(); vl, preds, labs = 0.0, [], []
            with torch.no_grad():
                for Xb, yb in loaders["val"]:
                    Xb, yb = Xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                    logits = model(Xb); vl += crit(logits, yb).item() * len(yb)
                    preds.append(logits.argmax(1).cpu()); labs.append(yb.cpu())
            preds, labs = torch.cat(preds).numpy(), torch.cat(labs).numpy()
            h = {"epoch": epoch, "train_loss": run / n, "val_loss": vl / len(labs), "val_acc": float((preds == labs).mean()),
                 "val_dir_f1": f1_score(labs, preds, labels=[0, 2], average="macro", zero_division=0)}
            hist.append(h)
            print(f"Ep {epoch:3d}/{epochs} | train {h['train_loss']:.4f} | val loss {h['val_loss']:.4f} | "
                  f"val acc {h['val_acc']:.4f} | val DirF1 {h['val_dir_f1']:.4f}")
            if best_state is None or h["val_loss"] < best:
                best, bad = h["val_loss"], 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    print(f"Early stop at epoch {epoch}"); break
        model.load_state_dict(best_state)
        return pd.DataFrame(hist)

    def train_torch(frame, model_name, feature_set, horizon, seed):
        hp = HPARAMS[model_name]
        tk = dict(hp["train_kwargs"])
        if EPOCHS_OVERRIDE: tk["epochs"] = EPOCHS_OVERRIDE
        set_seed(seed)
        cols = feature_columns(feature_set)
        X = torch.from_numpy(frame[cols].to_numpy(dtype=np.float32, copy=True))
        y = torch.from_numpy(class_labels(frame, horizon))
        targets = {s: target_indices(frame, s, hp["seq_len"], horizon) for s in ["train", "val", "test"]}
        pin = DEVICE.type == "cuda"
        loaders = {s: DataLoader(WindowDataset(X, y, targets[s], hp["seq_len"]),
                                 batch_size=hp["batch_size"] * (1 if s == "train" else 2), shuffle=(s == "train"),
                                 num_workers=NUM_WORKERS, pin_memory=pin) for s in targets}
        counts = np.bincount(y[targets["train"]].numpy(), minlength=3).astype(float)
        class_w = torch.tensor(counts.sum() / (3 * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
        print(f"{model_name.upper()} | {feature_set} | {len(cols)} features | h={horizon}s | seed {seed} | "
              + " | ".join(f"{s} {len(t):,}" for s, t in targets.items()))
        model = TORCH_MODELS[model_name](len(cols), **hp["model_kwargs"]).to(DEVICE)
        hist = fit_torch(model, loaders, class_w, **tk)
        p_val, _ = predict_proba(model, loaders["val"])
        p_test, _ = predict_proba(model, loaders["test"])
        best_epoch = int(hist.loc[hist["val_loss"].idxmin(), "epoch"])
        return p_val, p_test, targets, hist, best_epoch

In [ ]:
def train_lgbm(frame, feature_set, horizon, seed):
    import lightgbm as lgb
    hp = HPARAMS["lgbm"]
    cols = feature_columns(feature_set)
    targets = {s: target_indices(frame, s, 1, horizon) for s in ["train", "val", "test"]}
    X_all, y_all = frame[cols].to_numpy(dtype=np.float32), class_labels(frame, horizon)
    X = {s: X_all[t] for s, t in targets.items()}; y = {s: y_all[t] for s, t in targets.items()}
    print(f"LGBM | {feature_set} | {len(cols)} features | h={horizon}s | seed {seed} | "
          + " | ".join(f"{s} {len(t):,}" for s, t in targets.items()))
    model = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced", random_state=seed, verbose=-1,
                               **hp["params"])
    evals = {}
    model.fit(X["train"], y["train"], eval_set=[(X["train"], y["train"]), (X["val"], y["val"])],
              eval_names=["train", "val"], eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(hp["early_stopping_rounds"], verbose=False),
                         lgb.log_evaluation(100), lgb.record_evaluation(evals)])
    hist = pd.DataFrame({"epoch": np.arange(1, len(evals["val"]["multi_logloss"]) + 1),
                         "train_loss": evals["train"]["multi_logloss"], "val_loss": evals["val"]["multi_logloss"]})
    return model.predict_proba(X["val"]), model.predict_proba(X["test"]), targets, hist, int(model.best_iteration_)

## 5 · Experiment runner
`run_experiment` trains one model, chooses θ and the hysteresis levels on validation,
evaluates the test split once, and stores everything in the database:
* `runs` – identifiers + metrics, keyed by `run_id`;
* `predictions` – P(Down/Flat/Up) for every validation and test target;
* `history` – training curve.

Runs whose `run_id` is already in `runs` are skipped, so every loop below is resumable.

In [ ]:
from sklearn.metrics import classification_report, f1_score

def make_run_id(scheme, fold, model, feature_set, horizon, seed):
    return f"{scheme}|fold{fold}|{model}|{feature_set}|h{horizon}|seed{seed}"

def evaluate(frame, p_val, p_test, targets, horizon, verbose=True):
    """Classification metrics + backtests; trading rules chosen on validation only."""
    y_test = class_labels(frame, horizon)[targets["test"]]
    y_pred = p_test.argmax(1)
    ret = frame[f"future_return_{horizon}"].to_numpy()
    sv, st = nonoverlap_steps(targets["val"], horizon), nonoverlap_steps(targets["test"], horizon)
    r_val, r_test = ret[targets["val"][sv]], ret[targets["test"][st]]
    pv, pt = p_val[sv], p_test[st]

    th = float(pick_best(sweep_thresholds(pv, r_val, horizon))["threshold"])
    hy = pick_best(sweep_hysteresis(pv, r_val, horizon))
    enter, exit_ = float(hy["enter"]), float(hy["exit"])
    bt = {"argmax": backtest(positions_from_probs(pt), r_test, FEE, horizon),
          "th": backtest(positions_from_probs(pt, th), r_test, FEE, horizon),
          "hy": backtest(positions_hysteresis(pt, enter, exit_), r_test, FEE, horizon),
          "bh": backtest(np.ones(len(st)), r_test, 0.0, horizon)}
    if verbose:
        print(classification_report(y_test, y_pred, labels=[0, 1, 2], target_names=CLASS_NAMES, digits=4, zero_division=0))
        print(f"Chosen on validation: threshold {th:.3f} | hysteresis {enter:.2f}/{exit_:.2f}")
        print(f"Test backtest: {len(st):,} non-overlapping {horizon}s steps, fee {FEE * 1e4:.1f} bp")
        for k, name in [("argmax", "Argmax"), ("th", f"Threshold {th:.3f}"),
                        ("hy", f"Hysteresis {enter:.2f}/{exit_:.2f}"), ("bh", "Buy & hold (no fee)")]:
            print(fmt_stats(name, bt[k]))
    metrics = {
        "test_acc": float((y_pred == y_test).mean()), "flat_acc": float((y_test == 1).mean()),
        "test_dir_f1": float(f1_score(y_test, y_pred, labels=[0, 2], average="macro", zero_division=0)),
        "argmax_return": bt["argmax"]["total_return"], "argmax_sharpe": bt["argmax"]["sharpe"],
        "argmax_turnover": bt["argmax"]["turnover"],
        "threshold": th, "th_return": bt["th"]["total_return"], "th_sharpe": bt["th"]["sharpe"],
        "th_mdd": bt["th"]["max_drawdown"], "th_turnover": bt["th"]["turnover"],
        "hy_enter": enter, "hy_exit": exit_, "hy_return": bt["hy"]["total_return"], "hy_sharpe": bt["hy"]["sharpe"],
        "hy_mdd": bt["hy"]["max_drawdown"], "hy_turnover": bt["hy"]["turnover"],
        "bh_return": bt["bh"]["total_return"], "n_test_steps": int(len(st)),
    }
    return metrics, bt

def save_run(run_id, ident, metrics, p_val, p_test, targets, hist=None):
    """Predictions and history first, the `runs` row last: a row in `runs` means the run is complete."""
    db_exec("DELETE FROM history WHERE run_id = ?", [run_id])
    if db_has_table("predictions"):
        db_exec("DELETE FROM predictions WHERE run_id = ?", [run_id])
    preds = pd.concat([pd.DataFrame({"run_id": run_id, "split": s, "row_id": targets[s].astype(np.int64),
                                     "p_down": p[:, 0].astype("float32"), "p_flat": p[:, 1].astype("float32"),
                                     "p_up": p[:, 2].astype("float32")})
                       for s, p in [("val", p_val), ("test", p_test)]])
    db_write("predictions", preds)
    if hist is not None and len(hist):
        db_write("history", hist.assign(run_id=run_id)[["run_id"] + list(hist.columns)])
    db_write("runs", pd.DataFrame([{"run_id": run_id, **ident, **metrics, "created_at": pd.Timestamp.now(tz="UTC")}]))

def run_experiment(frame, model, feature_set, horizon=HORIZON, seed=SEED, scheme="main", fold=-1, test_day=""):
    run_id = make_run_id(scheme, fold, model, feature_set, horizon, seed)
    if run_id in done_run_ids():
        print("skip (already in database):", run_id); return run_id
    print("=" * 100 + f"\n{run_id}\n" + "=" * 100)
    p_val, p_test, targets, hist, best_epoch = (train_lgbm(frame, feature_set, horizon, seed) if model == "lgbm"
                                                else train_torch(frame, model, feature_set, horizon, seed))
    metrics, _ = evaluate(frame, p_val, p_test, targets, horizon)
    ident = {"scheme": scheme, "fold": fold, "test_day": test_day, "model": model, "feature_set": feature_set,
             "horizon": horizon, "seed": str(seed), "best_epoch": best_epoch}
    save_run(run_id, ident, metrics, p_val, p_test, targets, hist)
    gc.collect()
    return run_id

def load_predictions(run_id, split):
    p = db_query("SELECT row_id, p_down, p_flat, p_up FROM predictions WHERE run_id = ? AND split = ? ORDER BY row_id",
                 [run_id, split])
    return p["row_id"].to_numpy(), p[["p_down", "p_flat", "p_up"]].to_numpy()

def run_ensemble(frame, model, feature_set, horizon, seeds, scheme, fold, test_day=""):
    """Average the seeds' probabilities and evaluate them as seed = 'ens'."""
    run_id = make_run_id(scheme, fold, model, feature_set, horizon, "ens")
    if run_id in done_run_ids():
        return run_id
    parts = [make_run_id(scheme, fold, model, feature_set, horizon, s) for s in seeds]
    val = [load_predictions(r, "val") for r in parts]; test = [load_predictions(r, "test") for r in parts]
    targets = {"val": val[0][0], "test": test[0][0]}
    p_val, p_test = np.mean([v[1] for v in val], axis=0), np.mean([t[1] for t in test], axis=0)
    print(f"--- seed ensemble ({len(seeds)} seeds) | {scheme} fold {fold} | {model} | {feature_set} ---")
    metrics, _ = evaluate(frame, p_val, p_test, targets, horizon)
    ident = {"scheme": scheme, "fold": fold, "test_day": test_day, "model": model, "feature_set": feature_set,
             "horizon": horizon, "seed": "ens", "best_epoch": -1}
    save_run(run_id, ident, metrics, p_val, p_test, targets)
    return run_id

### Walk-forward folds
24-hour blocks from the first timestamp. Fold *k* trains on blocks *k … k+4*, validates on
*k+5* and tests on *k+6* (six test days). Labels are purged per fold and the PCA is refitted on
each fold's training days. `fold_frame` builds a separate frame, so `df` (main split) is untouched.

In [ ]:
def make_folds(frame, train_days=5):
    t0, day = frame["time"].iloc[0], pd.Timedelta(days=1)
    n_blocks = int(np.ceil((frame["time"].iloc[-1] - t0) / day))
    return [{"fold": k, "train_start": t0 + k * day, "val_start": t0 + (k + train_days) * day,
             "test_start": t0 + (k + train_days + 1) * day, "test_end": t0 + (k + train_days + 2) * day}
            for k in range(n_blocks - train_days - 1)]

def fold_frame(frame, fold, horizon=HORIZON):
    keep = ["time", f"label_{horizon}", f"future_return_{horizon}"] + LOB_COLS + MARKET_COLS + CLASSIC_COLS
    f = frame[keep].copy()
    t = f["time"]
    split = np.full(len(f), "none", dtype=object)
    for s, a, b in [("train", "train_start", "val_start"), ("val", "val_start", "test_start"),
                    ("test", "test_start", "test_end")]:
        split[((t >= fold[a]) & (t < fold[b])).to_numpy()] = s
    f["split"] = split
    label_end = (t + pd.Timedelta(seconds=horizon)).reset_index(drop=True)
    has_label = f[f"future_return_{horizon}"].notna().to_numpy()
    is_t = np.zeros(len(f), dtype=bool)
    for s, end in [("train", fold["val_start"]), ("val", fold["test_start"]), ("test", fold["test_end"])]:
        m = split == s
        is_t[m] = has_label[m] & (label_end[m] < end).to_numpy()
    f[f"is_target_{horizon}"] = is_t
    scaler, pca, _ = fit_pca(f.loc[split == "train", LOB_COLS], n_keep=len(PCA_COLS))
    f[PCA_COLS] = pca.transform(scaler.transform(f[LOB_COLS]))[:, :len(PCA_COLS)].astype("float32")
    counts = {s: int(((split == s) & is_t).sum()) for s in ["train", "val", "test"]}
    print(f"Fold {fold['fold']}: train {fold['train_start']:%m-%d %H:%M} → val {fold['val_start']:%m-%d %H:%M} → "
          f"test {fold['test_start']:%m-%d %H:%M} | targets {counts}")
    return f

### Import results computed before this notebook (run once)
The experiments in the README were first run with an earlier, script-based version of this
code, which stored features in `data/features.parquet` and results as CSV/NPZ files under
`results/`. This section copies them into the database, so the analysis and report sections
work without retraining. Set `STEPS["import_legacy"] = True` (the earlier feature table is
imported in Section 2 as well).

In [ ]:
def import_legacy(root=WORK_DIR):
    res = os.path.join(root, "results")
    done = done_run_ids()

    def metrics_of(r):
        return {c: (float(r[c]) if c in r and pd.notna(r[c]) else None) for c in METRIC_COLS} | {"n_test_steps": None}

    def add(ident, r, npz=None, probs=None):
        """probs: optional (p_val, p_test, targets) when the predictions are computed here."""
        run_id = make_run_id(ident["scheme"], ident["fold"], ident["model"], ident["feature_set"],
                             ident["horizon"], ident["seed"])
        if run_id in done:
            return
        if probs is None and npz is not None and os.path.exists(npz):
            d = np.load(npz)
            probs = (d["val_probs"], d["test_probs"], {"val": d["val_rows"], "test": d["test_rows"]})
        if probs is not None:
            save_run(run_id, ident, metrics_of(r), *probs)
        else:
            db_write("runs", pd.DataFrame([{"run_id": run_id, **ident, **metrics_of(r),
                                            "created_at": pd.Timestamp.now(tz="UTC")}]))
        done.add(run_id)

    # main split, per horizon (with predictions)
    for h in HORIZONS:
        hd = os.path.join(res, f"h{h}")
        for csv in sorted(f for f in (os.listdir(hd) if os.path.isdir(hd) else []) if f.endswith("_summary.csv")):
            for r in pd.read_csv(os.path.join(hd, csv)).to_dict("records"):
                ident = {"scheme": "main", "fold": -1, "test_day": "", "model": r["model"], "feature_set": r["feature_set"],
                         "horizon": h, "seed": str(r.get("seed", SEED)), "best_epoch": int(r.get("best_epoch") or -1)}
                add(ident, r, os.path.join(hd, "preds", f"{r['model']}_{r['feature_set']}.npz"))
    # first 10 s runs without a horizon folder (no hysteresis / predictions)
    for csv in sorted(f for f in (os.listdir(res) if os.path.isdir(res) else []) if f.endswith("_summary.csv")):
        for r in pd.read_csv(os.path.join(res, csv)).to_dict("records"):
            ident = {"scheme": "main", "fold": -1, "test_day": "", "model": r["model"], "feature_set": r["feature_set"],
                     "horizon": HORIZON, "seed": str(SEED), "best_epoch": int(r.get("best_epoch") or -1)}
            add(ident, r)
    # walk-forward
    wd = os.path.join(res, "walk_forward", f"h{HORIZON}")
    for csv in sorted(f for f in (os.listdir(wd) if os.path.isdir(wd) else []) if f.endswith("_runs.csv")):
        runs = pd.read_csv(os.path.join(wd, csv))
        runs["seed"] = runs["seed"].astype(str)
        for r in runs[runs["seed"] != "ens"].to_dict("records"):
            ident = {"scheme": "wf", "fold": int(r["fold"]), "test_day": r["test_day"], "model": r["model"],
                     "feature_set": r["feature_set"], "horizon": HORIZON, "seed": r["seed"],
                     "best_epoch": int(r.get("best_epoch") or -1)}
            add(ident, r, os.path.join(wd, "preds", f"{r['model']}_{r['feature_set']}_fold{int(r['fold'])}_seed{r['seed']}.npz"))
        for r in runs[runs["seed"] == "ens"].to_dict("records"):   # ensembles: average the seeds' predictions
            seeds = sorted(set(runs[(runs["fold"] == r["fold"]) & (runs["feature_set"] == r["feature_set"])
                                    & (runs["seed"] != "ens")]["seed"]))
            files = [os.path.join(wd, "preds", f"{r['model']}_{r['feature_set']}_fold{int(r['fold'])}_seed{s}.npz") for s in seeds]
            ident = {"scheme": "wf", "fold": int(r["fold"]), "test_day": r["test_day"], "model": r["model"],
                     "feature_set": r["feature_set"], "horizon": HORIZON, "seed": "ens", "best_epoch": -1}
            if not all(os.path.exists(f) for f in files):
                continue
            d = [np.load(f) for f in files]
            add(ident, r, probs=(np.mean([x["val_probs"] for x in d], 0), np.mean([x["test_probs"] for x in d], 0),
                                 {"val": d[0]["val_rows"], "test": d[0]["test_rows"]}))
    display(db_overview())

if STEPS["import_legacy"]:
    import_legacy()

## 6 · Run experiments
### 6.1 Main split — every model × feature set, 10-second horizon

In [ ]:
if STEPS["main_split"]:
    for m in MODELS:
        for fs in FEATURE_SETS:
            run_experiment(df, m, fs, HORIZON)

### 6.2 Prediction horizons — Full feature set at 30 / 60 / 300 s (LightGBM: all feature sets)

In [ ]:
if STEPS["horizons"]:
    for h in (30, 60, 300):
        for m in MODELS:
            for fs in (FEATURE_SETS if m == "lgbm" else ["full"]):
                run_experiment(df, m, fs, h)

### 6.3 Walk-forward — CNN (3 seeds + ensemble) and LightGBM, 10 s
On Colab the CNN needs about 30 minutes per fold on a T4 GPU, and LightGBM about 8 minutes
per run on the CPU.

In [ ]:
WF_PLAN = {"cnn": (["full"], [42, 43, 44]), "lgbm": (["full", "no_pca"], [42])}

if STEPS["walk_forward"]:
    for fold in make_folds(df):
        todo = [(m, fs, s) for m, (sets, seeds) in WF_PLAN.items() if m in MODELS
                for fs in sets for s in seeds + ["ens"]
                if make_run_id("wf", fold["fold"], m, fs, HORIZON, s) not in done_run_ids()]
        if not todo:
            continue
        fr = fold_frame(df, fold)
        day = f"{fold['test_start']:%m-%d %H:%M}"
        for m, (sets, seeds) in WF_PLAN.items():
            if m not in MODELS:
                continue
            for fs in sets:
                for s in seeds:
                    run_experiment(fr, m, fs, HORIZON, seed=s, scheme="wf", fold=fold["fold"], test_day=day)
                run_ensemble(fr, m, fs, HORIZON, seeds, "wf", fold["fold"], day)
        del fr; gc.collect()

## 7 · Analysis
All checks use the stored predictions — no retraining.
* **Robustness** (main-split runs): P&L from long vs short positions, P&L per day, and the test
  return over the 3×3 grid of neighbouring (enter, exit) values.
* **Execution delay** (all 10-second runs): the position for the step starting at second *t*
  is decided from the prediction made at *t − d*.
* **BTC/ETH timestamps**: how often the ETH snapshot inside a joined second is later than BTC.

In [ ]:
def runs_table():
    return db_query("SELECT * FROM runs ORDER BY scheme, fold, horizon, model, feature_set, seed")

RETURNS = {h: df[f"future_return_{h}"].to_numpy() for h in HORIZONS}
DAY = df["time"].dt.strftime("%m-%d").to_numpy()

def robustness_rows(runs):
    out = []
    for run in runs[runs["scheme"] == "main"].to_dict("records"):
        if pd.isna(run.get("hy_enter")):
            continue
        rows, probs = load_predictions(run["run_id"], "test")
        if len(rows) == 0:
            continue
        h, enter, exit_ = int(run["horizon"]), run["hy_enter"], run["hy_exit"]
        st, step_rows = test_steps(rows, h)
        ret = RETURNS[h][step_rows]
        pos = positions_hysteresis(probs[st], enter, exit_)
        net = backtest(pos, ret, FEE, h)["net"]
        rec = {"run_id": run["run_id"], "long_pnl": net[pos > 0].sum(), "short_pnl": net[pos < 0].sum(),
               "flat_fees": net[pos == 0].sum(), "pct_long": (pos > 0).mean(), "pct_short": (pos < 0).mean()}
        days = DAY[step_rows]
        rec["day_pnl"] = json.dumps({dd: float(net[days == dd].sum()) for dd in sorted(set(days))})
        sens = [backtest(positions_hysteresis(probs[st], round(enter + de, 2), round(exit_ + dx, 2)), ret, FEE, h)
                ["total_return"] for de in (-0.05, 0, 0.05) for dx in (-0.05, 0, 0.05)
                if 0 <= round(exit_ + dx, 2) < round(enter + de, 2) < 1]
        rec.update(sens_min=float(np.min(sens)), sens_median=float(np.median(sens)), sens_max=float(np.max(sens)))
        out.append(rec)
    return pd.DataFrame(out)

def delay_rows(runs, delays=(0, 1, 2, 3, 5)):
    out = []
    for run in runs[runs["horizon"] == HORIZON].to_dict("records"):
        if pd.isna(run.get("hy_enter")):
            continue
        rows, probs = load_predictions(run["run_id"], "test")
        if len(rows) == 0:
            continue
        where = dict(zip(rows.tolist(), range(len(rows))))
        for d in delays:
            ok = rows[np.array([t - d in where for t in rows])]
            step_rows = ok[nonoverlap_steps(ok, HORIZON)]
            p = probs[[where[t - d] for t in step_rows]]
            pos = positions_hysteresis(p, run["hy_enter"], run["hy_exit"])
            out.append({"run_id": run["run_id"], "delay_s": d,
                        "hy_return": backtest(pos, RETURNS[HORIZON][step_rows], FEE, HORIZON)["total_return"]})
    return pd.DataFrame(out)

def timestamp_check(nrows=300_000):
    raw = find_raw_dir(RAW_DIR)
    read = lambda f: pd.read_csv(os.path.join(raw, f), usecols=["system_time"], nrows=nrows)
    b, e = read("BTC_1sec.csv"), read("ETH_1sec.csv")
    for x in (b, e):
        x["t"] = pd.to_datetime(x["system_time"], utc=True); x["key"] = x["t"].dt.floor("s")
    m = b.drop_duplicates("key", keep="last").merge(e.drop_duplicates("key", keep="last"), on="key",
                                                    suffixes=("_btc", "_eth"))
    lag = (m["t_eth"] - m["t_btc"]).dt.total_seconds()
    return pd.DataFrame([{"matched_seconds": len(m), "mean_lag_s": lag.mean(), "median_lag_s": lag.median(),
                          "max_lag_s": lag.max(), "share_eth_later": (lag > 0).mean(),
                          "share_eth_later_0_1s": (lag > 0.1).mean()}])

In [ ]:
if STEPS["analysis"] and db_has_table("runs"):
    runs = runs_table()
    rob = robustness_rows(runs)
    if len(rob): db_write("robustness", rob, replace=True)
    dly = delay_rows(runs)
    if len(dly): db_write("delay", dly, replace=True)
    if not db_has_table("timestamp_check"):
        try:
            db_write("timestamp_check", timestamp_check(), replace=True)
        except Exception as err:          # raw CSVs not available here
            print("timestamp check skipped:", err)
db_overview()

## 8 · Report
Tables for the README / report, two figures, and CSV snapshots in `EXPORT_DIR`
(copy them to `results/` and `report/figures/` in the repository).

In [ ]:
if STEPS["report"] and db_has_table("runs"):
    os.makedirs(EXPORT_DIR, exist_ok=True)
    runs = runs_table()
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

    main10 = runs[(runs["scheme"] == "main") & (runs["horizon"] == HORIZON)]
    print("Main split, 10 s — classification and trading")
    display(main10.set_index(["model", "feature_set"])[
        ["test_acc", "flat_acc", "test_dir_f1", "argmax_return", "threshold", "th_return", "th_turnover",
         "hy_enter", "hy_exit", "hy_return", "hy_turnover", "bh_return"]].round(4))

    hor = runs[(runs["scheme"] == "main")].sort_values(["horizon", "model", "feature_set"])
    print("Prediction horizons")
    display(hor.set_index(["horizon", "model", "feature_set"])[
        ["flat_acc", "test_dir_f1", "argmax_return", "hy_return", "hy_turnover", "bh_return"]].round(4))
    hor.to_csv(os.path.join(EXPORT_DIR, "main_split_runs.csv"), index=False)

    wf = runs[runs["scheme"] == "wf"]
    if len(wf):
        print("Walk-forward — hysteresis return per test day")
        piv = wf.pivot_table(index=["model", "feature_set", "seed"], columns="test_day", values="hy_return")
        display(piv.assign(mean=piv.mean(axis=1)).round(4))
        wf.to_csv(os.path.join(EXPORT_DIR, "walk_forward_runs.csv"), index=False)

    for t in ("robustness", "delay", "timestamp_check"):
        if db_has_table(t):
            tab = db_query(f"SELECT * FROM {t}")
            tab.to_csv(os.path.join(EXPORT_DIR, f"{t}.csv"), index=False)
    if db_has_table("delay"):
        dly = db_query("SELECT d.*, r.scheme, r.model, r.feature_set, r.seed FROM delay d JOIN runs r USING (run_id)")
        print("Execution delay — mean hysteresis test return")
        display(dly.pivot_table(index=["scheme", "model", "feature_set", "seed"], columns="delay_s",
                                values="hy_return", aggfunc="mean").round(4))

In [ ]:
# Figures (same palette and style as the report)
BLUE, ORANGE, AQUA, YELLOW, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8985"
INK2, GRID = "#52514e", "#e4e3df"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.edgecolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False})
PCT = plt.matplotlib.ticker.PercentFormatter(1.0)

if STEPS["report"] and db_has_table("runs"):
    runs = runs_table()
    wf = runs[runs["scheme"] == "wf"]
    if len(wf):
        ens = wf[(wf["model"] == "cnn") & (wf["seed"] == "ens")].sort_values("fold")
        lgb_np = wf[(wf["model"] == "lgbm") & (wf["feature_set"] == "no_pca") & (wf["seed"] == "42")].sort_values("fold")
        fig, ax = plt.subplots(figsize=(6.4, 2.9))
        x, w = np.arange(len(ens)), 0.26
        for k, (name, vals, color) in enumerate([("CNN (3-seed ensemble)", ens["hy_return"], BLUE),
                                                 ("LightGBM (no_pca)", lgb_np["hy_return"], ORANGE),
                                                 ("Buy & hold", ens["bh_return"], GRAY)]):
            if len(vals) == len(ens):
                ax.bar(x + (k - 1) * w, vals, w, color=color, edgecolor="white", linewidth=1.2, label=name)
        ax.axhline(0, color=INK2, linewidth=0.8)
        ax.set_xticks(x, [f"Fold {f}\n{d[:5]}" for f, d in zip(ens["fold"], ens["test_day"])])
        ax.yaxis.set_major_formatter(PCT); ax.set_ylabel("Test-day return (hysteresis, 1.5 bp)")
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=8)
        fig.savefig(os.path.join(EXPORT_DIR, "fig_walk_forward.pdf"), bbox_inches="tight"); plt.show()

    if db_has_table("delay"):
        dly = db_query("SELECT d.*, r.scheme, r.model, r.feature_set, r.seed FROM delay d JOIN runs r USING (run_id)")
        fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.7))
        left = dly[(dly["scheme"] == "wf") & (dly["model"] == "cnn") & (dly["seed"] == "ens")]
        if len(left):
            g = left.groupby("delay_s")["hy_return"].mean()
            axes[0].plot(g.index, g.values, color=BLUE, lw=2, marker="o", ms=4.5, mec="white")
        axes[0].set_title("CNN ensemble, walk-forward (mean of 6 days)", fontsize=9)
        for lbl, m, fs, c in [("CNN", "cnn", "full", BLUE), ("LightGBM", "lgbm", "no_pca", ORANGE),
                              ("LSTM", "lstm", "full", AQUA), ("Transformer", "transformer", "full", YELLOW)]:
            g = dly[(dly["scheme"] == "main") & (dly["model"] == m) & (dly["feature_set"] == fs)].sort_values("delay_s")
            if len(g):
                axes[1].plot(g["delay_s"], g["hy_return"], color=c, lw=2, marker="o", ms=4.5, mec="white", label=lbl)
        axes[1].set_title("Main split (test 04-17 to 04-19)", fontsize=9); axes[1].legend(fontsize=8)
        for a in axes:
            a.axhline(0, color=INK2, lw=0.8); a.set_xticks([0, 1, 2, 3, 5]); a.set_xlabel("Execution delay (seconds)")
            a.yaxis.set_major_formatter(PCT)
        axes[0].set_ylabel("Hysteresis test return")
        fig.tight_layout(); fig.savefig(os.path.join(EXPORT_DIR, "fig_delay.pdf"), bbox_inches="tight"); plt.show()
    print("Exports in", EXPORT_DIR, ":", sorted(os.listdir(EXPORT_DIR)))